In [1]:
import time
import boto3
import requests

transcribe = boto3.client("transcribe")



In [2]:
job_name = f"job-{int(time.time())}"


In [ ]:
VOCAB_NAME = VOCAB  # Name of vocabulary to be created
s3_bucket = BUCKET  # Bucket name on S3
media_uri = WAV_DIR  # Path to the .wave file in S3 Bucket

# Upload the vocabulary list to the S3 bucket
transcribe.create_vocabulary(
    VocabularyName=VOCAB_NAME,
    LanguageCode="en-US",
    VocabularyFileUri=f"s3://{s3_bucket}/Transcribe/VocabularyList.txt"
)

# Checking to see if vocabulary is created.
while True:
    v = transcribe.get_vocabulary(VocabularyName=VOCAB_NAME)
    state = v["VocabularyState"]
    print("Vocab state:", state)
    if state in ("READY", "FAILED"):
        break
    time.sleep(5)

In [ ]:
transcribe.start_transcription_job(
    TranscriptionJobName=job_name,
    Media={"MediaFileUri": media_uri},
    MediaFormat="wav",
    LanguageCode="en-US",
    OutputBucketName=BUCKET,
    Settings={
        "VocabularyName": VOCAB_NAME
    },
    OutputKey=f"transcribe-output/{job_name}.json",
)

while True:
    job = transcribe.get_transcription_job(TranscriptionJobName=job_name)
    status = job["TranscriptionJob"]["TranscriptionJobStatus"]
    if status in ("COMPLETED", "FAILED"):
        print("Status:", status)
        break
    print("Waiting... current status:", status)
    time.sleep(10)

transcript_uri = job["TranscriptionJob"]["Transcript"]["TranscriptFileUri"]
transcript_uri

In [ ]:
transcribe.delete_transcription_job(TranscriptionJobName=job_name)
